In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q openai datasets scikit-learn
import os, re, json, time, pickle
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/llama70b_strategy"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"

# ===== STRATEGYCLAIM LOADER — need to match your DeepSeek/Gemma runs =====
# TODO: paste your exact loader here
# baf_df must have columns: claim, gold (1=true/0=false), example_index
# balanced 250 True / 250 False, random_state=42

print("First claim:", baf_df.iloc[0]["claim"][:70])
print("Loaded:", len(baf_df), "| True:", baf_df["gold"].sum())

In [ ]:
!pip install -q openai datasets scikit-learn
import os, re, json, time, pickle, subprocess, sys
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_from_disk

WORK_DIR = Path("/kaggle/working/llama70b_strategy"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"

# clone the ArgLLM repo (contains the StrategyQA dataset)
if not os.path.exists("/kaggle/working/argumentative-llms"):
    subprocess.run(["git","clone","https://github.com/CLArg-group/argumentative-llms.git",
                    "/kaggle/working/argumentative-llms"], capture_output=True, text=True)

dataset_path = "/kaggle/working/argumentative-llms/Datasets/StrategyQA/Experiment"
strategy_experiment = load_from_disk(dataset_path)
df = strategy_experiment.to_pandas().reset_index(drop=True)

def format_facts(facts):
    if facts is None: return ""
    if isinstance(facts, (list, tuple, np.ndarray)):
        return "\n".join(f"- {str(f)}" for f in facts)
    return str(facts)

baf_df = pd.DataFrame({
    "example_index": range(len(df)),
    "claim": df["claim"].astype(str),
    "facts_text": df["facts"].apply(format_facts),
    "gold": df["valid"].apply(lambda v: int(bool(v))),
})
print("Number of examples:", len(baf_df))
print("Gold distribution:", baf_df["gold"].value_counts().sort_index().to_dict())
print("First claim:", baf_df.iloc[0]["claim"][:80])

In [ ]:
BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5, "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55, "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55, "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2, "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0, "max_retries": 5,
}
def clip01(v):
    try: return max(0.0,min(1.0,float(v)))
    except: return 0.0
def extract_json_object(text):
    text=str(text or "").strip(); text=re.sub(r"```(?:json)?","",text,flags=re.IGNORECASE).strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    s=text.find("{"); e=text.rfind("}")
    if s>=0 and e>s: return json.loads(text[s:e+1])
    raise ValueError("No JSON")
def deepseek_json_call(prompt, temperature, max_tokens, retries=5):
    last=None
    for a in range(1,retries+1):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You are an expert in claim verification, formal argumentation and strict JSON output. You always output valid JSON."},
                          {"role":"user","content":prompt}],
                temperature=temperature,max_tokens=max_tokens)
            t=resp.choices[0].message.content
            if not t: raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last=e
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(5); continue
            if a<retries: time.sleep(2*a)
    raise RuntimeError(f"failed: {last}")
print("✅ CELL 1b ready")

In [ ]:
def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw=deepseek_json_call(prompt,config["generation_temperature"],1500,config["max_retries"])
    out={"interpreted_claim":str(raw.get("interpreted_claim","")).strip(),"pro":[],"con":[]}
    for role in ("pro","con"):
        for e in (raw.get(role,[]) if isinstance(raw.get(role,[]),list) else []):
            if not isinstance(e,dict): continue
            t=str(e.get("text","")).strip()
            if not t: continue
            out[role].append({"text":t,"relevance":clip01(e.get("relevance")),"grounding":clip01(e.get("grounding")),"clarity":clip01(e.get("clarity")),"overall_quality":clip01(e.get("overall_quality"))})
    return out

def select_arguments(gen, config=BAF_CONFIG):
    nodes={"CLAIM":{"role":"CLAIM","text":""}}; scored=[]
    for src,role,pre,mx in [("pro","PRO","P",config["max_pro_keep"]),("con","CON","N",config["max_con_keep"])]:
        cands=[]
        for it in gen.get(src,[]):
            rl,g=clip01(it.get("relevance")),clip01(it.get("grounding")); cl,q=clip01(it.get("clarity")),clip01(it.get("overall_quality"))
            rank=0.30*rl+0.30*g+0.15*cl+0.25*q; sc={**it,"role":role,"rank_score":rank}; scored.append(sc)
            if rl>=config["min_relevance"] and g>=config["min_grounding"] and cl>=config["min_clarity"] and q>=config["min_quality"]: cands.append(sc)
        cands.sort(key=lambda x:x["rank_score"],reverse=True)
        for n,it in enumerate(cands[:mx],1):
            nodes[f"{pre}{n}"]={"role":role,"text":it["text"],"relevance":it["relevance"],"grounding":it["grounding"],"clarity":it["clarity"],"overall_quality":it["overall_quality"],"rank_score":it["rank_score"]}
    return {"nodes":nodes,"all_scored":scored}

def build_relation_candidates(nodes):
    pro=[n for n,i in nodes.items() if i["role"]=="PRO"]; con=[n for n,i in nodes.items() if i["role"]=="CON"]; c=[]
    for p in pro: c.append({"source":p,"target":"CLAIM","allowed":["SUPPORT","NONE"]})
    for n in con: c.append({"source":n,"target":"CLAIM","allowed":["ATTACK","NONE"]})
    for p in pro:
        for n in con:
            c.append({"source":p,"target":n,"allowed":["ATTACK","NONE"]}); c.append({"source":n,"target":p,"allowed":["ATTACK","NONE"]})
    return c
print("✅ CELL 2 ready")

In [ ]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG, batch_size=16):
    cands=build_relation_candidates(nodes)
    at="\n".join(f"{nid} [{i['role']}]: {i['text']}" for nid,i in nodes.items() if nid!="CLAIM")
    aud=[]
    for bs in range(0,len(cands),batch_size):
        b=cands[bs:bs+batch_size]
        pb="\n".join(f"{it['source']} -> {it['target']} | allowed={it['allowed']}" for it in b)
        prompt=f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw=deepseek_json_call(prompt,config["audit_temperature"],1800,config["max_retries"])
        exp={(it["source"],it["target"]) for it in b}
        allow={(it["source"],it["target"]):set(it["allowed"]) for it in b}
        ret={}
        for r in (raw.get("relations",[]) if isinstance(raw.get("relations",[]),list) else []):
            if not isinstance(r,dict): continue
            s,t=str(r.get("source","")).strip(),str(r.get("target","")).strip()
            typ=str(r.get("type","NONE")).strip().upper(); pr=(s,t)
            if pr not in exp or typ not in allow[pr]: continue
            ret[pr]={"source":s,"target":t,"type":typ,"target_match":clip01(r.get("target_match")),"grounding":clip01(r.get("grounding")),"relation_score":clip01(r.get("relation_score")),"reason":str(r.get("reason","")).strip()}
        for it in b:
            pr=(it["source"],it["target"])
            if pr not in ret:
                ret[pr]={"source":pr[0],"target":pr[1],"type":"NONE","target_match":0.0,"grounding":0.0,"relation_score":0.0,"reason":"omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations":aud}
print("✅ CELL 3 ready")

In [ ]:
def relation_passes(r,config=BAF_CONFIG): return r["target_match"]>=config["min_target_match"] and r["grounding"]>=config["min_grounding_edge"] and r["relation_score"]>=config["min_relation_score"]
def relation_lookup(rels): return {(r["source"],r["target"]):r for r in rels}
def select_formal_supports(rels,config=BAF_CONFIG): return sorted(set((r["source"],r["target"]) for r in rels if r["type"]=="SUPPORT" and r["target"]=="CLAIM" and r["source"].startswith("P") and relation_passes(r,config)))
def select_con_claim_attacks(rels,config=BAF_CONFIG):
    p=[r for r in rels if r["type"]=="ATTACK" and r["target"]=="CLAIM" and r["source"].startswith("N") and relation_passes(r,config)]
    p.sort(key=lambda x:x["relation_score"],reverse=True); return p[:config["max_claim_attacks"]]
def select_independent_conflict_pairs(rels,config=BAF_CONFIG):
    lk=relation_lookup(rels); cand=[]
    pro=sorted({r["source"] for r in rels if r["source"].startswith("P")}); con=sorted({r["source"] for r in rels if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc,cp=lk.get((p,n)),lk.get((n,p))
            if not pc or not cp: continue
            if pc["type"]=="ATTACK" and cp["type"]=="ATTACK" and relation_passes(pc,config) and relation_passes(cp,config):
                cand.append({"pro":p,"con":n,"pair_score":min(pc["relation_score"],cp["relation_score"])})
    cand.sort(key=lambda x:x["pair_score"],reverse=True); sel=[]; up=set(); uc=set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc: continue
        sel.append(c); up.add(c["pro"]); uc.add(c["con"])
        if len(sel)>=config["max_conflict_pairs"]: break
    return sel,cand
def support_closure(nodes,sup):
    cl=set(sup); ch=True
    while ch:
        ch=False; new=set()
        for a,b in cl:
            for c,d in cl:
                if b==c and (a,d) not in cl: new.add((a,d))
        if new: cl|=new; ch=True
    return cl
def derive_attacks(nodes,direct,sup):
    sc=support_closure(nodes,sup); sa=set(); se=set()
    for s,sd in sc:
        for at,t in direct:
            if sd==at: sa.add((s,t))
    for at,ad in direct:
        for s,sd in sc:
            if ad==s: se.add((at,sd))
    eff=(set(direct)|sa|se); eff={e for e in eff if e[0]!=e[1]}
    return {"support_closure":sorted(sc),"supported_attacks":sorted(sa),"secondary_attacks":sorted(se),"effective_attacks":sorted(eff)}
print("✅ CELL 4 ready")

In [ ]:
def all_subsets(items):
    il=sorted(items)
    for sz in range(len(il)+1):
        for sub in combinations(il,sz): yield set(sub)
def cf(e,att): return not any(s in e and t in e for s,t in att)
def att_of(a,att): return {s for s,t in att if t==a}
def ext_att(e,a,att): return any(s in e and t==a for s,t in att)
def defends(e,a,att): return all(ext_att(e,x,att) for x in att_of(a,att))
def admissible(e,att): return cf(e,att) and all(defends(e,a,att) for a in e)
def preferred_ext(args,att):
    adm=[s for s in all_subsets(args) if admissible(s,att)]
    return [c for c in adm if not any(c<o for o in adm)]
def stable_ext(args,att):
    out=[]
    for c in all_subsets(args):
        if not cf(c,att): continue
        if all(ext_att(c,a,att) for a in args-c): out.append(c)
    return out
def norm(exts): return [sorted(e) for e in exts]
def decisions(exts,cid="CLAIM"):
    if not exts: return {"credulous":0,"skeptical":0,"majority":0}
    cc=sum(cid in e for e in exts)
    return {"credulous":int(cc>=1),"skeptical":int(cc==len(exts)),"majority":int(cc*2>=len(exts))}
def rebuild(inter,config=BAF_CONFIG):
    nodes=dict(inter["nodes"]); rels=inter["all_audited_relations"]; sup=set(select_formal_supports(rels,config))
    sca=select_con_claim_attacks(rels,config); sp,_=select_independent_conflict_pairs(rels,config); direct=set()
    for r in sca: direct.add((r["source"],r["target"]))
    for p in sp: direct.add((p["pro"],p["con"])); direct.add((p["con"],p["pro"]))
    conn={"CLAIM"}
    for st in sup|direct: conn.add(st[0]); conn.add(st[1])
    nodes={k:v for k,v in nodes.items() if k in conn}; vids=set(nodes)
    sup={e for e in sup if e[0] in vids and e[1] in vids}; direct={e for e in direct if e[0] in vids and e[1] in vids}
    der=derive_attacks(vids,direct,sup); eff=set(map(tuple,der["effective_attacks"]))
    pref=preferred_ext(vids,eff); stab=stable_ext(vids,eff)
    return {**inter,"nodes":nodes,"supports":sorted(sup),"selected_claim_attacks":sca,
        "selected_conflict_pairs":[{"pro":p["pro"],"con":p["con"],"pair_score":p["pair_score"]} for p in sp],
        "direct_attacks":sorted(direct),"support_closure":der["support_closure"],"supported_attacks":der["supported_attacks"],
        "secondary_attacks":der["secondary_attacks"],"effective_attacks":der["effective_attacks"],
        "preferred_extensions":norm(pref),"stable_extensions":norm(stab),"preferred_decisions":decisions(pref),"stable_decisions":decisions(stab)}
def run_v3_baf_example(row,config=BAF_CONFIG):
    claim=str(row["claim"]).strip(); facts=str(row["facts_text"]).strip(); gold=int(row["gold"])
    gen=generate_and_score_arguments(claim,facts,config); sel=select_arguments(gen,config); nodes=sel["nodes"]; nodes["CLAIM"]["text"]=claim
    audit=audit_baf_relations(claim,facts,nodes,config,config["audit_batch_size"])
    inter={"example_index":int(row["example_index"]),"claim":claim,"facts_text":facts,"gold":gold,"interpreted_claim":gen["interpreted_claim"],"generated_arguments":gen,"argument_scores":sel["all_scored"],"nodes":nodes,"all_audited_relations":audit["all_audited_relations"],"model":MODEL_NAME,"config":dict(config)}
    return rebuild(inter,config)
print("✅ CELL 5 ready — pipeline complete")

In [ ]:
CKPT = WORK_DIR / "llama70b_strategy_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))
def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (Llama 3.3 70B StrategyClaim)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]   # facts_text is ALREADY in the row (pre-loaded)
        try:
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 20 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done\n")

run_batch(100)

results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
from collections import Counter
print(f"=== Llama 3.3 70B StrategyClaim at {len(results)} ===")
for rule in ["credulous","skeptical","majority"]:
    p=[r["preferred_decisions"][rule] for r in results]
    print(f"  Extension {rule}: {accuracy_score(gold,p)*100:.1f}%")
print(f"  net_support: {accuracy_score(gold,[net_support_pred(r) for r in results])*100:.1f}%")
print(f"  extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"llama70b_strategy_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"llama70b_strategy_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"llama70b_strategy_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="llama70b_strategy_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download llama70b_strategy_{N}_backup.zip</a>'))
print(f"Backed up {N}")

In [ ]:
run_batch(100)

results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
from collections import Counter
print(f"=== Llama 3.3 70B StrategyClaim at {len(results)} ===")
for rule in ["credulous","skeptical","majority"]:
    p=[r["preferred_decisions"][rule] for r in results]
    print(f"  Extension {rule}: {accuracy_score(gold,p)*100:.1f}%")
print(f"  net_support: {accuracy_score(gold,[net_support_pred(r) for r in results])*100:.1f}%")
print(f"  extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"llama70b_strategy_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"llama70b_strategy_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"llama70b_strategy_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="llama70b_strategy_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download llama70b_strategy_{N}_backup.zip</a>'))
print(f"Backed up {N}")

In [ ]:
run_batch(100)

results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
from collections import Counter
print(f"=== Llama 3.3 70B StrategyClaim at {len(results)} ===")
for rule in ["credulous","skeptical","majority"]:
    p=[r["preferred_decisions"][rule] for r in results]
    print(f"  Extension {rule}: {accuracy_score(gold,p)*100:.1f}%")
print(f"  net_support: {accuracy_score(gold,[net_support_pred(r) for r in results])*100:.1f}%")
print(f"  extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"llama70b_strategy_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"llama70b_strategy_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"llama70b_strategy_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="llama70b_strategy_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download llama70b_strategy_{N}_backup.zip</a>'))
print(f"Backed up {N}")

In [ ]:
run_batch(100)

results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
from collections import Counter
print(f"=== Llama 3.3 70B StrategyClaim at {len(results)} ===")
for rule in ["credulous","skeptical","majority"]:
    p=[r["preferred_decisions"][rule] for r in results]
    print(f"  Extension {rule}: {accuracy_score(gold,p)*100:.1f}%")
print(f"  net_support: {accuracy_score(gold,[net_support_pred(r) for r in results])*100:.1f}%")
print(f"  extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"llama70b_strategy_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"llama70b_strategy_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"llama70b_strategy_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="llama70b_strategy_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download llama70b_strategy_{N}_backup.zip</a>'))
print(f"Backed up {N}")

In [ ]:
run_batch(100)

import base64, zipfile
from collections import Counter
from IPython.display import HTML, display

ck = load_ck()
results = [ck[k] for k in sorted(ck)]
gold = [r["gold"] for r in results]
N = len(results)

def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s, t) in r["effective_attacks"] if t == "CLAIM"))

rows = []
for sem in ["preferred", "stable"]:
    key = f"{sem}_decisions"
    if not results[0].get(key):
        continue
    for rule in results[0][key]:
        p = [r[key][rule] for r in results]
        tn, fp, fn, tp = confusion_matrix(gold, p).ravel()
        rows.append({"method": f"{sem}_{rule}", "n": N,
                     "accuracy": accuracy_score(gold, p),
                     "precision": precision_score(gold, p, zero_division=0),
                     "recall": recall_score(gold, p, zero_division=0),
                     "f1": f1_score(gold, p, zero_division=0),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp})

ns = [net_support_pred(r) for r in results]
tn, fp, fn, tp = confusion_matrix(gold, ns).ravel()
rows.append({"method": "net_support", "n": N,
             "accuracy": accuracy_score(gold, ns),
             "precision": precision_score(gold, ns, zero_division=0),
             "recall": recall_score(gold, ns, zero_division=0),
             "f1": f1_score(gold, ns, zero_division=0),
             "TN": tn, "FP": fp, "FN": fn, "TP": tp})

print(f"=== Llama 3.3 70B StrategyClaim — FINAL, ALL rules ({N} examples) ===\n")
print(pd.DataFrame(rows).sort_values("accuracy", ascending=False)
        .round(3).reset_index(drop=True).to_string())
print(f"\nGold balance: True={sum(gold)} False={N - sum(gold)}")
print(f"Extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

pkl = WORK_DIR / f"llama70b_strategy_{N}.pkl"; pickle.dump(results, open(pkl, "wb"))
zp = WORK_DIR / f"llama70b_strategy_{N}_backup.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pkl, arcname=f"llama70b_strategy_{N}.pkl")
b64 = base64.b64encode(open(zp, "rb").read()).decode()
display(HTML(f'<a download="llama70b_strategy_{N}_backup.zip" href="data:application/zip;base64,{b64}" '
             f'style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;'
             f'border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download {N} backup</a>'))
print(f"Backed up {N}")

In [ ]:
ck = load_ck()
results = [ck[k] for k in sorted(ck)]
gold = [r["gold"] for r in results]
n = len(results)

def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    return {"method": name, "n": n,
        "accuracy": round(accuracy_score(gold,pred),3),
        "precision": round(precision_score(gold,pred,zero_division=0),3),
        "recall": round(recall_score(gold,pred,zero_division=0),3),
        "f1": round(f1_score(gold,pred,zero_division=0),3),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp}

def no_objection_pred(r, exts_key):
    exts = r[exts_key]
    if not exts: return 0
    eff = set(map(tuple, r["effective_attacks"]))
    for ext in exts:
        es = set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff):
            return 1
    return 0

def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))

rows = []
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"preferred_{rule}", [r["preferred_decisions"][rule] for r in results]))
rows.append(metrics_row("preferred_no_objection", [no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"stable_{rule}", [r["stable_decisions"][rule] for r in results]))
rows.append(metrics_row("stable_no_objection", [no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(metrics_row("net_support", [net_support_pred(r) for r in results]))

table = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
print(f"=== Llama 3.3 70B StrategyClaim — FINAL, ALL rules ({n} examples) ===\n")
print(table.to_string())
print(f"\nGold balance: True={sum(gold)} False={n-sum(gold)}")

In [ ]:
from collections import Counter
ck = load_ck(); results = [ck[k] for k in sorted(ck)]
n = len(results)

for key in ["preferred_extensions", "stable_extensions"]:
    if key not in results[0]: continue
    c = Counter(len(r[key]) for r in results)
    print(f"\n{key.split('_')[0].upper()} extensions per example:")
    for k in sorted(c):
        print(f"  {k} extension(s): {c[k]:3d} examples ({c[k]/n*100:.1f}%)")
    print(f"  → avg: {sum(len(r[key]) for r in results)/n:.2f} | max: {max(c)}")

cp = Counter(len(r.get("conflict_pairs", [])) for r in results)
print("\nConflict pairs per example (n → 2^n extensions):")
for k in sorted(cp):
    print(f"  {k} pair(s): {cp[k]:3d} ({cp[k]/n*100:.1f}%) → expect {2**k} ext")

for lbl, fld in [("PRO", "pro_kept"), ("CON", "con_kept")]:
    if fld in results[0]:
        c = Counter(len(r[fld]) for r in results)
        print(f"\n{lbl} arguments kept per example:")
        for k in sorted(c):
            print(f"  {k} {lbl}: {c[k]:3d} ({c[k]/n*100:.1f}%)")

In [ ]:
from collections import Counter
import numpy as np
ck = load_ck(); results = [ck[k] for k in sorted(ck)]
n = len(results)

print(f"=== Llama 3.3 70B StrategyClaim — Extension Statistics ({n} examples) ===")

for key in ["preferred_extensions", "stable_extensions"]:
    c = Counter(len(r[key]) for r in results)
    print(f"\n{key.split('_')[0].upper()} extensions per example:")
    for k in sorted(c):
        print(f"  {k} extension(s): {c[k]:3d} examples ({c[k]/n*100:.1f}%)")
    print(f"  → avg: {np.mean([len(r[key]) for r in results]):.2f} | max: {max(c)}")

cp = Counter(len(r.get("selected_conflict_pairs", [])) for r in results)
print("\nConflict pairs per example (n → 2^n extensions):")
for k in sorted(cp):
    print(f"  {k} pair(s): {cp[k]:3d} ({cp[k]/n*100:.1f}%) → expect {2**k} ext")

for lbl, pre in [("PRO", "P"), ("CON", "N")]:
    c = Counter(sum(1 for nid in r["nodes"] if nid.startswith(pre)) for r in results)
    print(f"\n{lbl} arguments kept per example:")
    for k in sorted(c):
        print(f"  {k} {lbl}: {c[k]:3d} ({c[k]/n*100:.1f}%)")

print(f"\nAvg supports (edges): {np.mean([len(r['supports']) for r in results]):.2f}")
print(f"Avg attacks (edges): {np.mean([len(r['effective_attacks']) for r in results]):.2f}")

In [ ]:
def graph_size(r):
    return sum(1 for nid in r["nodes"] if nid.startswith(("P","N")))

empty = [r for r in results if graph_size(r) == 0]
rich  = [r for r in results if graph_size(r) >= 2]
for name, sub in [("empty graph (0 args)", empty), ("2+ args", rich)]:
    if not sub: continue
    g = [r["gold"] for r in sub]
    p = [r["preferred_decisions"]["skeptical"] for r in sub]
    print(f"{name}: n={len(sub)} | skeptical acc={accuracy_score(g,p)*100:.1f}% "
          f"| gold True={sum(g)} ({sum(g)/len(sub)*100:.0f}%)")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

ck = load_ck(); results = [ck[k] for k in sorted(ck)]

def npro(r): return sum(1 for n in r["nodes"] if n.startswith("P"))
def ncon(r): return sum(1 for n in r["nodes"] if n.startswith("N"))

buckets = {
    "2-ext (conflict)":  [r for r in results if len(r["preferred_extensions"]) == 2],
    "1-ext PRO+CON":     [r for r in results if len(r["preferred_extensions"]) == 1 and npro(r) > 0 and ncon(r) > 0],
    "1-ext PRO only":    [r for r in results if len(r["preferred_extensions"]) == 1 and npro(r) > 0 and ncon(r) == 0],
    "1-ext CON only":    [r for r in results if len(r["preferred_extensions"]) == 1 and npro(r) == 0 and ncon(r) > 0],
    "EMPTY (0 args)":    [r for r in results if npro(r) == 0 and ncon(r) == 0],
}
for k, v in buckets.items():
    print(f"{k}: {len(v)}")

def draw_graph(r, ax, tag):
    G = nx.DiGraph(); colors = []
    for nid in r["nodes"]:
        G.add_node(nid)
        colors.append("#FFD700" if nid == "CLAIM" else "#90EE90" if nid.startswith("P") else "#FF9999")
    sup = [tuple(e) for e in r["supports"]]
    att = [tuple(e) for e in r["effective_attacks"]]
    G.add_edges_from(sup); G.add_edges_from(att)
    pos = nx.spring_layout(G, seed=42, k=1.8)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=700, ax=ax)
    nx.draw_networkx_labels(G, pos, {n: n.replace("CLAIM","C") for n in G.nodes}, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=sup, edge_color="green", width=2, arrows=True, ax=ax, connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_edges(G, pos, edgelist=att, edge_color="red", width=2, style="dashed", arrows=True, ax=ax, connectionstyle="arc3,rad=0.12")
    gold = "T" if r["gold"]==1 else "F"
    cred = "T" if r["preferred_decisions"]["credulous"]==1 else "F"
    skep = "T" if r["preferred_decisions"]["skeptical"]==1 else "F"
    cp = len(r.get("selected_conflict_pairs", []))
    ok = "✓" if (r["preferred_decisions"]["skeptical"] == r["gold"]) else "✗"
    ax.set_title(f"{tag} {ok}\n{len(r['preferred_extensions'])} ext | {cp} conflict | "
                 f"gold={gold} cr={cred} sk={skep} | {npro(r)}PRO {ncon(r)}CON", fontsize=8)
    ax.axis("off")

# 2 from each bucket = 10 graphs
show = [(r, k) for k, v in buckets.items() for r in v[:2]]

fig, axes = plt.subplots(5, 2, figsize=(12, 24))
for i, ax in enumerate(axes.flat):
    if i < len(show): draw_graph(show[i][0], ax, show[i][1])
    else: ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/llama70b_ext_graphs.png", dpi=100, bbox_inches="tight")
plt.show()
print("\nYellow=CLAIM, green=PRO, red=CON | green=support, red dashed=attack")

# text detail for the same 10
for r, tag in show:
    print(f"\n=== [{tag}] {r['claim'][:70]}")
    print(f"    gold={'T' if r['gold']==1 else 'F'} | pairs: {[(p['pro'],p['con']) for p in r.get('selected_conflict_pairs',[])]}")
    for nid in sorted(r["nodes"]):
        if nid.startswith(("P","N")):
            print(f"    {nid}: {r['nodes'][nid]['text'][:70]}")
    for j, ext in enumerate(r["preferred_extensions"], 1):
        print(f"    Ext{j}: {sorted(ext)} {'[CLAIM in]' if 'CLAIM' in ext else '[CLAIM out]'}")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

ck = load_ck(); results = [ck[k] for k in sorted(ck)]

def npro(r): return sum(1 for n in r["nodes"] if n.startswith("P"))
def ncon(r): return sum(1 for n in r["nodes"] if n.startswith("N"))
def complexity(r):
    # nodes + edges, with extensions and conflicts weighted heavily
    return (len(r["preferred_extensions"]) * 10
            + len(r.get("selected_conflict_pairs", [])) * 5
            + npro(r) + ncon(r)
            + len(r["supports"]) + len(r["effective_attacks"]))

ranked = sorted(results, key=complexity, reverse=True)
show = ranked[:8]

print("Top 8 by complexity:")
for i, r in enumerate(show, 1):
    print(f"  [{i}] score={complexity(r):3d} | {len(r['preferred_extensions'])} ext | "
          f"{npro(r)}P {ncon(r)}N | {len(r['supports'])} sup | "
          f"{len(r['effective_attacks'])} att | {len(r.get('selected_conflict_pairs',[]))} pairs")

def draw_graph(r, ax, rank):
    G = nx.DiGraph(); colors = []
    for nid in r["nodes"]:
        G.add_node(nid)
        colors.append("#FFD700" if nid == "CLAIM" else "#90EE90" if nid.startswith("P") else "#FF9999")
    sup = [tuple(e) for e in r["supports"]]
    att = [tuple(e) for e in r["effective_attacks"]]
    direct = set(map(tuple, r.get("direct_attacks", [])))
    derived = [e for e in att if e not in direct] if direct else []
    G.add_edges_from(sup); G.add_edges_from(att)
    pos = nx.spring_layout(G, seed=42, k=2.0)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=800, ax=ax)
    nx.draw_networkx_labels(G, pos, {n: n.replace("CLAIM","C") for n in G.nodes}, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=sup, edge_color="green", width=2,
                           arrows=True, ax=ax, connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_edges(G, pos, edgelist=[e for e in att if e not in derived],
                           edge_color="red", width=2, style="dashed",
                           arrows=True, ax=ax, connectionstyle="arc3,rad=0.12")
    if derived:
        nx.draw_networkx_edges(G, pos, edgelist=derived, edge_color="purple", width=1.5,
                               style="dotted", arrows=True, ax=ax, connectionstyle="arc3,rad=0.25")
    gold = "T" if r["gold"]==1 else "F"
    cred = "T" if r["preferred_decisions"]["credulous"]==1 else "F"
    skep = "T" if r["preferred_decisions"]["skeptical"]==1 else "F"
    ok = "✓" if r["preferred_decisions"]["skeptical"] == r["gold"] else "✗"
    ax.set_title(f"#{rank} score={complexity(r)} {ok}\n"
                 f"{len(r['preferred_extensions'])} ext | {len(r.get('selected_conflict_pairs',[]))} pairs | "
                 f"gold={gold} cr={cred} sk={skep}\n{npro(r)}PRO {ncon(r)}CON | "
                 f"{len(sup)} sup {len(att)} att", fontsize=8)
    ax.axis("off")

fig, axes = plt.subplots(4, 2, figsize=(13, 22))
for i, ax in enumerate(axes.flat):
    if i < len(show): draw_graph(show[i], ax, i+1)
    else: ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/llama70b_complex_graphs.png", dpi=100, bbox_inches="tight")
plt.show()
print("\nYellow=CLAIM green=PRO red=CON | green=support, red dashed=direct attack, purple dotted=derived attack")

for i, r in enumerate(show, 1):
    print(f"\n{'='*70}\n[{i}] {r['claim'][:80]}")
    print(f"gold={'T' if r['gold']==1 else 'F'} | cred={r['preferred_decisions']['credulous']} "
          f"skep={r['preferred_decisions']['skeptical']}")
    for nid in sorted(r["nodes"]):
        if nid.startswith(("P","N")):
            print(f"  {nid}: {r['nodes'][nid]['text'][:80]}")
    print(f"SUPPORTS: {r['supports']}")
    print(f"DIRECT ATTACKS: {r.get('direct_attacks', '(not stored)')}")
    print(f"EFFECTIVE ATTACKS: {r['effective_attacks']}")
    print(f"CONFLICT PAIRS: {[(p['pro'],p['con']) for p in r.get('selected_conflict_pairs',[])]}")
    for j, ext in enumerate(r["preferred_extensions"], 1):
        print(f"  Ext{j}: {sorted(ext)} {'[CLAIM in]' if 'CLAIM' in ext else '[CLAIM out]'}")

In [ ]:
ck = load_ck(); results = [ck[k] for k in sorted(ck)]
multi  = [r for r in results if len(r["preferred_extensions"]) > 1]
single = [r for r in results if len(r["preferred_extensions"]) == 1]
print("--- Accuracy split (the semantics finding) ---")
for name, sub in [(f"multi-extension ({len(multi)})", multi), (f"single-extension ({len(single)})", single)]:
    g = [r["gold"] for r in sub]
    for rule in ["credulous","skeptical","majority"]:
        p = [r["preferred_decisions"][rule] for r in sub]
        print(f"  {name} — {rule}: {accuracy_score(g,p)*100:.1f}%")
    print()

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
multi = [r for r in results if len(r["preferred_extensions"]) > 1]
g = np.array([r["gold"] for r in multi])
c = np.array([r["preferred_decisions"]["credulous"] for r in multi]) == g
s = np.array([r["preferred_decisions"]["skeptical"] for r in multi]) == g
tbl = [[int((c&s).sum()), int((c&~s).sum())], [int((~c&s).sum()), int((~c&~s).sum())]]
print(tbl, mcnemar(tbl, exact=True))

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
multi = [r for r in results if len(r["preferred_extensions"]) > 1]
g = np.array([r["gold"] for r in multi])
c = np.array([r["preferred_decisions"]["credulous"] for r in multi]) == g
s = np.array([r["preferred_decisions"]["skeptical"] for r in multi]) == g
tbl = [[int((c&s).sum()), int((c&~s).sum())], [int((~c&s).sum()), int((~c&~s).sum())]]
print(tbl, mcnemar(tbl, exact=True))

In [ ]:
import glob, os
for pat in ["/kaggle/input/**/*.pkl", "/kaggle/working/**/*.pkl"]:
    for f in sorted(glob.glob(pat, recursive=True)):
        print(f"{os.path.getsize(f)/1e6:7.1f} MB  {f}")

In [ ]:
!pip install -q openai datasets scikit-learn

import os, re, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

WORK_DIR = Path("/kaggle/working/dfquad_llama_strategyclaim")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT = WORK_DIR / "dfquad_llama_strategyclaim_facts_ckpt.pkl"

client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"),
                base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # SAME model as the V2 extension notebook

if not Path("/kaggle/working/argumentative-llms").exists():
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk("/kaggle/working/argumentative-llms/Datasets/StrategyQA/Experiment").to_pandas().reset_index(drop=True)

def format_facts(facts):
    if facts is None: return ""
    if isinstance(facts, (list, tuple, np.ndarray)):
        return "\n".join(f"- {str(f)}" for f in facts)
    return str(facts)

baf_df = pd.DataFrame({
    "example_index": range(len(df)),
    "claim": df["claim"].astype(str),
    "facts_text": df["facts"].apply(format_facts),
    "gold": df["valid"].apply(lambda v: int(bool(v))),
})
print("Loaded:", len(baf_df), "| True:", baf_df["gold"].sum(), "| first:", baf_df.iloc[0]["claim"][:60])

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/argumentative-llms")
try:
    import Uncertainpy.src.uncertainpy.gradual as grad
    agg_f = grad.semantics.modular.ProductAggregation()
    inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)
    HAVE_UNCERTAINPY = True
except Exception as e:
    print("Uncertainpy not importable, using built-in DF-QuAD only:", e)
    HAVE_UNCERTAINPY = False

def llm_raw(prompt, max_tokens=200, retries=8):
    last = None
    for a in range(1, retries+1):
        try:
            r = client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"user","content":prompt}],
                temperature=0.7, top_p=0.95, max_tokens=max_tokens)
            t = r.choices[0].message.content
            if t: return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e; msg = str(e)
            time.sleep(20*a if any(c in msg for c in ["429","502","503","504"]) or "rate" in msg.lower() else 3*a)
    raise RuntimeError(f"failed: {last}")

def am_prompt(claim, facts, kind):            # kind: "supporting" / "attacking"
    verb = "support" if kind == "supporting" else "attack"
    cond = ("there is a non-zero probability that the claim is true" if kind == "supporting"
            else "there is a non-zero probability that the claim is false")
    return (f"Facts:\n{facts}\n\n"
            f"Please provide a single short argument {kind} the following claim. "
            f"Construct the argument so it refers to the truthfulness of the claim. "
            f"Only provide an argument if you think there is a valid and convincing {verb} "
            f"for this claim ({cond}), otherwise return: N/A.\n"
            f"Claim: {claim}\nArgument:")

def ue_arg_prompt(claim, facts, arg, kind):
    return (f"Facts:\n{facts}\n\n"
            f"Consider the following claim and an argument {kind} it.\n"
            f"Claim: {claim}\nArgument: {arg}\n"
            f"How strong is this argument, on a scale from 0 (very weak) to 1 (very strong)? "
            f"Answer with a single number between 0 and 1 and nothing else.")

def ue_topic_prompt(claim, facts):
    return (f"Facts:\n{facts}\n\n"
            f"Claim: {claim}\n"
            f"How likely is this claim to be true, on a scale from 0 (certainly false) to 1 (certainly true)? "
            f"Answer with a single number between 0 and 1 and nothing else.")

def gen_argument(claim, facts, kind):
    t = llm_raw(am_prompt(claim, facts, kind), max_tokens=120)
    t = t.replace("Argument:", "").strip().strip('"')
    return "N/A" if (not t or "N/A" in t[:10].upper()) else t

def get_lik(prompt):
    t = llm_raw(prompt, max_tokens=10)
    m = re.search(r"(?<![\d.])(0(?:\.\d+)?|1(?:\.0+)?|\.\d+)(?![\d.])", t)
    return float(m.group(1)) if m else 0.5    # default 0.5, as in ArgLLM

def dfquad_manual(tau, sup, att):
    vs = sup if sup is not None else 0.0
    va = att if att is not None else 0.0
    if va >= vs: return tau - tau * (va - vs)
    return tau + (1 - tau) * (vs - va)

def dfquad_strength(v, est):
    tau = v["topic_base"] if est else 0.5
    sup = None if v["sup"] == "N/A" else float(v["sup_base"])
    att = None if v["att"] == "N/A" else float(v["att_base"])
    s = dfquad_manual(float(tau), sup, att)
    if HAVE_UNCERTAINPY:
        bag = grad.BAG()
        claim = grad.Argument("db0", v["claim"], float(tau)); bag.arguments["db0"] = claim
        if sup is not None: bag.add_support(grad.Argument("S1", "s", sup), claim)
        if att is not None: bag.add_attack(grad.Argument("A1", "a", att), claim)
        grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
        s_lib = float(bag.arguments["db0"].strength)
        if abs(s_lib - s) > 1e-6: print(f"⚠ mismatch {s:.4f} vs {s_lib:.4f}")
        s = s_lib
    return s

# quick test on one claim
r0 = baf_df.iloc[0]
v = {"claim": r0.claim}
v["sup"] = gen_argument(r0.claim, r0.facts_text, "supporting")
v["att"] = gen_argument(r0.claim, r0.facts_text, "attacking")
v["sup_base"] = get_lik(ue_arg_prompt(r0.claim, r0.facts_text, v["sup"], "supporting")) if v["sup"] != "N/A" else None
v["att_base"] = get_lik(ue_arg_prompt(r0.claim, r0.facts_text, v["att"], "attacking")) if v["att"] != "N/A" else None
v["topic_base"] = get_lik(ue_topic_prompt(r0.claim, r0.facts_text))
print(v); print("0.5 Base:", dfquad_strength(v, False), "| Est. Base:", dfquad_strength(v, True))
print("✅ CELL 2 ready")

In [ ]:
results = pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}
print("Resuming from", len(results), "done")

for i, r in baf_df.iterrows():
    if i in results: continue
    try:
        v = {"claim": r.claim, "gold": int(r.gold)}
        v["sup"] = gen_argument(r.claim, r.facts_text, "supporting")
        v["att"] = gen_argument(r.claim, r.facts_text, "attacking")
        v["sup_base"] = get_lik(ue_arg_prompt(r.claim, r.facts_text, v["sup"], "supporting")) if v["sup"] != "N/A" else None
        v["att_base"] = get_lik(ue_arg_prompt(r.claim, r.facts_text, v["att"], "attacking")) if v["att"] != "N/A" else None
        v["topic_base"] = get_lik(ue_topic_prompt(r.claim, r.facts_text))
        v["s_05"] = dfquad_strength(v, False); v["s_est"] = dfquad_strength(v, True)
        results[i] = v
    except Exception as e:
        print(f"❌ {i}: {e}")
    if len(results) % 10 == 0:
        pickle.dump(results, open(CKPT, "wb"))
        print(f"{len(results)}/{len(baf_df)}")
pickle.dump(results, open(CKPT, "wb"))

# ---- results ----
res = pd.DataFrame.from_dict(results, orient="index").sort_index()
y = res["gold"].values
for name, col in [("DF-QuAD 0.5 Base", "s_05"), ("DF-QuAD Est. Base", "s_est")]:
    p = (res[col] > 0.5).astype(int).values
    print(f"\n{name}  (n={len(res)})")
    print(f"  Acc {100*accuracy_score(y,p):.1f} | P {100*precision_score(y,p,zero_division=0):.1f} "
          f"| R {100*recall_score(y,p,zero_division=0):.1f} | F1 {100*f1_score(y,p,zero_division=0):.1f}")
    print("  Confusion [[TN FP][FN TP]]:", confusion_matrix(y, p).tolist())
print("\nN/A supporters:", (res.sup == "N/A").sum(), "| N/A attackers:", (res.att == "N/A").sum())
res.to_pickle(WORK_DIR / "dfquad_llama_strategyclaim_facts_final.pkl")
print("Saved → download", WORK_DIR / "dfquad_llama_strategyclaim_facts_final.pkl")

New Df-Quad with 

In [ ]:
!pip install -q openai datasets scikit-learn

import os, re, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

MODEL = "llama"        # "deepseek" | "llama" | "gptoss" | "gemma"

SETUP = {
    "deepseek": ("DEEPSEEK_API_KEY", "https://api.deepseek.com",
                 "deepseek-chat"),
    "llama":    ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1",
                 "meta-llama/llama-3.3-70b-instruct"),
    "gptoss":   ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1",
                 "openai/gpt-oss-120b"),
    "gemma":    ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1",
                 "google/gemma-4-31b-it"),
}
secret, base_url, MODEL_NAME = SETUP[MODEL]
client = OpenAI(api_key=UserSecretsClient().get_secret(secret),
                base_url=base_url)

WORK_DIR = Path(f"/kaggle/working/baselines_{MODEL}")
WORK_DIR.mkdir(parents=True, exist_ok=True)

repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join("- " + str(x) for x in f)
    return "" if f is None else str(f)


claims = df["claim"].astype(str).tolist()
facts = df["facts"].apply(format_facts).tolist()
gold = df["valid"].apply(lambda v: int(bool(v))).tolist()
print(MODEL_NAME, "| claims:", len(claims), "| true:", sum(gold))


def call(prompt, temperature, max_tokens, top_p=None):
    kw = dict(model=MODEL_NAME, temperature=temperature,
              max_tokens=max_tokens,
              messages=[{"role": "user", "content": prompt}])
    if top_p is not None:
        kw["top_p"] = top_p
    last = None
    for a in range(1, 9):
        try:
            r = client.chat.completions.create(**kw)
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e
            busy = any(c in str(e) for c in ["429", "502", "503", "504"])
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def report(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": len(pred),
            "accuracy": round(accuracy_score(gold, pred) * 100, 1),
            "precision": round(precision_score(gold, pred,
                                               zero_division=0) * 100, 1),
            "recall": round(recall_score(gold, pred,
                                         zero_division=0) * 100, 1),
            "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

In [ ]:
def parse_tf(text):
    t = (text or "").strip().lower().lstrip("*\"' ")
    if t.startswith("true"):
        return 1
    if t.startswith("false"):
        return 0
    has_t, has_f = "true" in t, "false" in t
    if has_t and not has_f:
        return 1
    if has_f and not has_t:
        return 0
    return None


def ask(claim, fact=None):
    if fact:
        p = (f"Facts:\n{fact}\n\nUsing these facts, is the following "
             'statement true or false? Answer with only one word: '
             f'"true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    else:
        p = ('Is the following statement true or false? Answer with only '
             f'one word: "true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    try:
        return parse_tf(call(p, 0, 10))
    except Exception:
        return None


DCK = WORK_DIR / "direct_strict_ckpt.pkl"
dres = pickle.load(open(DCK, "rb")) if DCK.exists() else {}
print("already done:", len(dres))
for i in range(len(claims)):
    if i in dres:
        continue
    dres[i] = {"no_facts": ask(claims[i]),
               "with_facts": ask(claims[i], facts[i])}
    pickle.dump(dres, open(DCK, "wb"))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(claims)}")

rows = []
for k in ["no_facts", "with_facts"]:
    pred = [dres[i][k] if dres[i][k] is not None else 0
            for i in range(len(claims))]
    fails = sum(dres[i][k] is None for i in range(len(claims)))
    rows.append(report(f"direct_{k}", pred) | {"unparsed": fails})
print(pd.DataFrame(rows).to_string())

# ****NEW TEST VERSION 2****# 

In [ ]:
!pip install -q openai datasets scikit-learn

import os, re, json, time, pickle, shutil
from pathlib import Path
from itertools import combinations
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from IPython.display import FileLink, display

WORK_DIR = Path("/kaggle/working/llama_strategyclaim_v2")
WORK_DIR.mkdir(parents=True, exist_ok=True)

key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
client = OpenAI(api_key=key,
                base_url="https://openrouter.ai/api/v1",
                timeout=180)
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"

repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if f is None:
        return ""
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join(f"- {str(x)}" for x in f)
    return str(f)


baf_df = pd.DataFrame({
    "example_index": range(len(df)),
    "claim": df["claim"].astype(str),
    "facts_text": df["facts"].apply(format_facts),
    "gold": df["valid"].apply(lambda v: int(bool(v))),
})
print("claims:", len(baf_df), "| true:", baf_df["gold"].sum())

r = client.chat.completions.create(
    model=MODEL_NAME, temperature=0, max_tokens=10,
    messages=[{"role": "user",
               "content": "Is Paris in France? Answer true or false."}])
print("test:", r.choices[0].message.content)

In [ ]:
BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5,
    "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55,
    "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55,
    "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2,
    "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0,
    "max_retries": 5,
}

SYSTEM = ("You are an expert in claim verification, formal "
          "argumentation and strict JSON output.")


def clip01(v):
    try:
        return max(0.0, min(1.0, float(v)))
    except Exception:
        return 0.0


def extract_json_object(text):
    text = str(text or "").strip()
    text = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE)
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s >= 0 and e > s:
        return json.loads(text[s:e + 1])
    raise ValueError("No JSON")


def json_call(prompt, temperature, max_tokens, retries=8):
    last = None
    for a in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens)
            t = resp.choices[0].message.content
            if not t:
                raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last = e
            msg = str(e)
            if "402" in msg:
                raise RuntimeError("no OpenRouter credit left")
            busy = any(c in msg for c in ["429", "502", "503", "504"])
            if busy or "rate" in msg.lower():
                time.sleep(20 * a)
            else:
                time.sleep(3 * a)
    raise RuntimeError(f"failed: {last}")


def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw = json_call(prompt, config["generation_temperature"], 1900,
                    config["max_retries"])
    out = {"interpreted_claim":
           str(raw.get("interpreted_claim", "")).strip(),
           "pro": [], "con": []}
    for role in ("pro", "con"):
        items = raw.get(role, [])
        if not isinstance(items, list):
            items = []
        for e in items:
            if not isinstance(e, dict):
                continue
            t = str(e.get("text", "")).strip()
            if not t:
                continue
            out[role].append({
                "text": t,
                "relevance": clip01(e.get("relevance")),
                "grounding": clip01(e.get("grounding")),
                "clarity": clip01(e.get("clarity")),
                "overall_quality": clip01(e.get("overall_quality"))})
    return out


def select_arguments(gen, config=BAF_CONFIG):
    nodes = {"CLAIM": {"role": "CLAIM", "text": ""}}
    scored = []
    plan = [("pro", "PRO", "P", config["max_pro_keep"]),
            ("con", "CON", "N", config["max_con_keep"])]
    for src, role, pre, mx in plan:
        cands = []
        for it in gen.get(src, []):
            rl = clip01(it.get("relevance"))
            g = clip01(it.get("grounding"))
            cl = clip01(it.get("clarity"))
            q = clip01(it.get("overall_quality"))
            rank = 0.30 * rl + 0.30 * g + 0.15 * cl + 0.25 * q
            sc = {**it, "role": role, "rank_score": rank}
            scored.append(sc)
            ok = (rl >= config["min_relevance"]
                  and g >= config["min_grounding"]
                  and cl >= config["min_clarity"]
                  and q >= config["min_quality"])
            if ok:
                cands.append(sc)
        cands.sort(key=lambda x: x["rank_score"], reverse=True)
        for n, it in enumerate(cands[:mx], 1):
            nodes[f"{pre}{n}"] = {
                "role": role, "text": it["text"],
                "relevance": it["relevance"],
                "grounding": it["grounding"],
                "clarity": it["clarity"],
                "overall_quality": it["overall_quality"],
                "rank_score": it["rank_score"]}
    return {"nodes": nodes, "all_scored": scored}


def build_relation_candidates(nodes):
    pro = [n for n, i in nodes.items() if i["role"] == "PRO"]
    con = [n for n, i in nodes.items() if i["role"] == "CON"]
    c = []
    for p in pro:
        c.append({"source": p, "target": "CLAIM",
                  "allowed": ["SUPPORT", "NONE"]})
    for n in con:
        c.append({"source": n, "target": "CLAIM",
                  "allowed": ["ATTACK", "NONE"]})
    for p in pro:
        for n in con:
            c.append({"source": p, "target": n,
                      "allowed": ["ATTACK", "NONE"]})
            c.append({"source": n, "target": p,
                      "allowed": ["ATTACK", "NONE"]})
    return c


print("CELL 2 ready")

In [ ]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG,
                        batch_size=16):
    cands = build_relation_candidates(nodes)
    at = "\n".join(f"{nid} [{i['role']}]: {i['text']}"
                   for nid, i in nodes.items() if nid != "CLAIM")
    aud = []
    for bs in range(0, len(cands), batch_size):
        b = cands[bs:bs + batch_size]
        pb = "\n".join(f"{it['source']} -> {it['target']} | "
                       f"allowed={it['allowed']}" for it in b)
        prompt = f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw = json_call(prompt, config["audit_temperature"], 2200,
                        config["max_retries"])
        exp = {(it["source"], it["target"]) for it in b}
        allow = {(it["source"], it["target"]): set(it["allowed"])
                 for it in b}
        ret = {}
        rels = raw.get("relations", [])
        if not isinstance(rels, list):
            rels = []
        for r in rels:
            if not isinstance(r, dict):
                continue
            s = str(r.get("source", "")).strip()
            t = str(r.get("target", "")).strip()
            typ = str(r.get("type", "NONE")).strip().upper()
            pr = (s, t)
            if pr not in exp or typ not in allow[pr]:
                continue
            ret[pr] = {"source": s, "target": t, "type": typ,
                       "target_match": clip01(r.get("target_match")),
                       "grounding": clip01(r.get("grounding")),
                       "relation_score": clip01(r.get("relation_score")),
                       "reason": str(r.get("reason", "")).strip()}
        for it in b:
            pr = (it["source"], it["target"])
            if pr not in ret:
                ret[pr] = {"source": pr[0], "target": pr[1],
                           "type": "NONE", "target_match": 0.0,
                           "grounding": 0.0, "relation_score": 0.0,
                           "reason": "omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations": aud}


def relation_passes(r, config=BAF_CONFIG):
    return (r["target_match"] >= config["min_target_match"]
            and r["grounding"] >= config["min_grounding_edge"]
            and r["relation_score"] >= config["min_relation_score"])


def relation_lookup(rels):
    return {(r["source"], r["target"]): r for r in rels}


def select_formal_supports(rels, config=BAF_CONFIG):
    return sorted(set(
        (r["source"], r["target"]) for r in rels
        if r["type"] == "SUPPORT" and r["target"] == "CLAIM"
        and r["source"].startswith("P") and relation_passes(r, config)))


def select_con_claim_attacks(rels, config=BAF_CONFIG):
    p = [r for r in rels
         if r["type"] == "ATTACK" and r["target"] == "CLAIM"
         and r["source"].startswith("N") and relation_passes(r, config)]
    p.sort(key=lambda x: x["relation_score"], reverse=True)
    return p[:config["max_claim_attacks"]]


def select_independent_conflict_pairs(rels, config=BAF_CONFIG):
    lk = relation_lookup(rels)
    cand = []
    pro = sorted({r["source"] for r in rels
                  if r["source"].startswith("P")})
    con = sorted({r["source"] for r in rels
                  if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc, cp = lk.get((p, n)), lk.get((n, p))
            if not pc or not cp:
                continue
            if (pc["type"] == "ATTACK" and cp["type"] == "ATTACK"
                    and relation_passes(pc, config)
                    and relation_passes(cp, config)):
                cand.append({"pro": p, "con": n,
                             "pair_score": min(pc["relation_score"],
                                               cp["relation_score"])})
    cand.sort(key=lambda x: x["pair_score"], reverse=True)
    sel, up, uc = [], set(), set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc:
            continue
        sel.append(c)
        up.add(c["pro"])
        uc.add(c["con"])
        if len(sel) >= config["max_conflict_pairs"]:
            break
    return sel, cand


def support_closure(nodes, sup):
    cl = set(sup)
    ch = True
    while ch:
        ch = False
        new = set()
        for a, b in cl:
            for c, d in cl:
                if b == c and (a, d) not in cl:
                    new.add((a, d))
        if new:
            cl |= new
            ch = True
    return cl


def derive_attacks(nodes, direct, sup):
    sc = support_closure(nodes, sup)
    sa, se = set(), set()
    for s, sd in sc:
        for at, t in direct:
            if sd == at:
                sa.add((s, t))
    for at, ad in direct:
        for s, sd in sc:
            if ad == s:
                se.add((at, sd))
    eff = {e for e in (set(direct) | sa | se) if e[0] != e[1]}
    return {"support_closure": sorted(sc),
            "supported_attacks": sorted(sa),
            "secondary_attacks": sorted(se),
            "effective_attacks": sorted(eff)}


print("CELL 3 ready")

In [ ]:
def all_subsets(items):
    il = sorted(items)
    for sz in range(len(il) + 1):
        for sub in combinations(il, sz):
            yield set(sub)


def cf(e, att):
    return not any(s in e and t in e for s, t in att)


def att_of(a, att):
    return {s for s, t in att if t == a}


def ext_att(e, a, att):
    return any(s in e and t == a for s, t in att)


def defends(e, a, att):
    return all(ext_att(e, x, att) for x in att_of(a, att))


def admissible(e, att):
    return cf(e, att) and all(defends(e, a, att) for a in e)


def preferred_ext(args, att):
    adm = [s for s in all_subsets(args) if admissible(s, att)]
    return [c for c in adm if not any(c < o for o in adm)]


def stable_ext(args, att):
    return [c for c in all_subsets(args)
            if cf(c, att) and all(ext_att(c, a, att) for a in args - c)]


def norm(exts):
    return [sorted(e) for e in exts]


def decisions(exts, cid="CLAIM"):
    if not exts:
        return {"credulous": 0, "skeptical": 0, "majority": 0}
    cc = sum(cid in e for e in exts)
    return {"credulous": int(cc >= 1),
            "skeptical": int(cc == len(exts)),
            "majority": int(cc * 2 >= len(exts))}


def rebuild(inter, config=BAF_CONFIG):
    nodes = dict(inter["nodes"])
    rels = inter["all_audited_relations"]
    sup = set(select_formal_supports(rels, config))
    sca = select_con_claim_attacks(rels, config)
    sp, _ = select_independent_conflict_pairs(rels, config)
    direct = set()
    for r in sca:
        direct.add((r["source"], r["target"]))
    for p in sp:
        direct.add((p["pro"], p["con"]))
        direct.add((p["con"], p["pro"]))
    conn = {"CLAIM"}
    for st in sup | direct:
        conn.add(st[0])
        conn.add(st[1])
    nodes = {k: v for k, v in nodes.items() if k in conn}
    vids = set(nodes)
    sup = {e for e in sup if e[0] in vids and e[1] in vids}
    direct = {e for e in direct if e[0] in vids and e[1] in vids}
    der = derive_attacks(vids, direct, sup)
    eff = set(map(tuple, der["effective_attacks"]))
    pref = preferred_ext(vids, eff)
    stab = stable_ext(vids, eff)
    pairs = [{"pro": p["pro"], "con": p["con"],
              "pair_score": p["pair_score"]} for p in sp]
    return {**inter, "nodes": nodes, "supports": sorted(sup),
            "selected_claim_attacks": sca,
            "selected_conflict_pairs": pairs,
            "direct_attacks": sorted(direct),
            "support_closure": der["support_closure"],
            "supported_attacks": der["supported_attacks"],
            "secondary_attacks": der["secondary_attacks"],
            "effective_attacks": der["effective_attacks"],
            "preferred_extensions": norm(pref),
            "stable_extensions": norm(stab),
            "preferred_decisions": decisions(pref),
            "stable_decisions": decisions(stab)}


def run_example(row, config=BAF_CONFIG):
    claim = str(row["claim"]).strip()
    facts = str(row["facts_text"]).strip()
    gold = int(row["gold"])
    gen = generate_and_score_arguments(claim, facts, config)
    sel = select_arguments(gen, config)
    nodes = sel["nodes"]
    nodes["CLAIM"]["text"] = claim
    audit = audit_baf_relations(claim, facts, nodes, config,
                                config["audit_batch_size"])
    inter = {"example_index": int(row["example_index"]),
             "claim": claim, "facts_text": facts, "gold": gold,
             "interpreted_claim": gen["interpreted_claim"],
             "generated_arguments": gen,
             "argument_scores": sel["all_scored"],
             "nodes": nodes,
             "all_audited_relations": audit["all_audited_relations"],
             "model": MODEL_NAME, "pipeline_version": "v2",
             "config": dict(config)}
    return rebuild(inter, config)


# quick test on one example
test = run_example(baf_df.iloc[0])
print("gold", test["gold"],
      "| credulous", test["preferred_decisions"]["credulous"],
      "| nodes", len(test["nodes"]),
      "| extensions", len(test["preferred_extensions"]))

In [ ]:
CKPT = WORK_DIR / "llama_strategyclaim_v2_ckpt.pkl"
WORKERS = 4


def load_ck():
    return pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}


def save_ck(d):
    tmp = CKPT.with_suffix(".tmp")
    pickle.dump(d, open(tmp, "wb"))
    tmp.replace(CKPT)


def safe_one(i):
    try:
        return i, run_example(baf_df.iloc[i]), None
    except Exception as e:
        return i, None, str(e)[:80]


def run_pass(label):
    d = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in d]
    print(f"[{label}] have {len(d)}/500, running {len(todo)}",
          flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = [pool.submit(safe_one, i) for i in todo]
        for f in as_completed(futs):
            i, v, err = f.result()
            if err:
                print(f"  error at {i}: {err}", flush=True)
                continue
            d[i] = v
            save_ck(d)
            if len(d) % 25 == 0:
                acc = np.mean([d[k]["preferred_decisions"]["credulous"]
                               == d[k]["gold"] for k in d])
                mins = (time.time() - t0) / 60
                print(f"  {len(d)}/500 | {mins:.0f} min | "
                      f"credulous so far {acc * 100:.1f}%", flush=True)
    return load_ck()


d = run_pass("main run")
for r in range(2):
    if len(d) < len(baf_df):
        time.sleep(30)
        d = run_pass(f"retry {r + 1}")

# ===== results =====
results = [d[k] for k in sorted(d)]
gold = [r["gold"] for r in results]
n = len(results)


def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": n,
            "acc": round(accuracy_score(gold, pred) * 100, 1),
            "prec": round(precision_score(gold, pred,
                                          zero_division=0) * 100, 1),
            "rec": round(recall_score(gold, pred,
                                      zero_division=0) * 100, 1),
            "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def no_objection_pred(r, key):
    exts = [set(e) for e in r[key]]
    if not exts or not any("CLAIM" in e for e in exts):
        return 0
    att = {s for (s, t) in map(tuple, r["direct_attacks"])
           if t == "CLAIM"}
    return int(not (att & set().union(*exts)))


def net_support_pred(r):
    n_att = sum(1 for (s, t) in map(tuple, r["effective_attacks"])
                if t == "CLAIM")
    return int(len(r["supports"]) >= n_att)


rows = []
for sem in ["preferred", "stable"]:
    for rule in ["credulous", "skeptical", "majority"]:
        rows.append(metrics_row(
            f"{sem}_{rule}",
            [r[f"{sem}_decisions"][rule] for r in results]))
    rows.append(metrics_row(
        f"{sem}_no_objection",
        [no_objection_pred(r, f"{sem}_extensions") for r in results]))
rows.append(metrics_row("net_support",
                        [net_support_pred(r) for r in results]))

pd.set_option("display.width", 220)
print(f"\n=== FINAL: Llama 3.3 70B StrategyClaim - Version 2 "
      f"({n} claims) ===\n")
print(pd.DataFrame(rows).to_string(index=False))

# ===== extensions =====
n_ext = [len(r["preferred_extensions"]) for r in results]
cnt = Counter(n_ext)
multi = sum(k > 1 for k in n_ext)
print("\nextension counts:", dict(sorted(cnt.items())))
print("mean extensions:", round(float(np.mean(n_ext)), 2))
print(f"more than 1 extension: {multi} ({multi / n * 100:.1f}%)")
ext_rows = []
for k in sorted(cnt):
    sub = [r for r, e in zip(results, n_ext) if e == k]
    ext_rows.append({
        "n_ext": k, "claims": len(sub),
        "percent": round(len(sub) / n * 100, 1),
        "credulous_acc": round(np.mean(
            [r["preferred_decisions"]["credulous"] == r["gold"]
             for r in sub]) * 100, 1),
        "skeptical_acc": round(np.mean(
            [r["preferred_decisions"]["skeptical"] == r["gold"]
             for r in sub]) * 100, 1)})
print(pd.DataFrame(ext_rows).to_string(index=False))
print("failed claims:", 500 - n)

# ===== backup =====
os.chdir("/kaggle/working")
out = f"llama_strategyclaim_v2_{n}.pkl"
pickle.dump(results, open(out, "wb"))
pd.DataFrame(rows).to_csv("llama_strategyclaim_v2_results.csv",
                          index=False)
try:
    display(FileLink(out))
    display(FileLink("llama_strategyclaim_v2_results.csv"))
except Exception:
    print("download from the Output panel:", out)

In [ ]:
print('ok')

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

LABEL = "Llama 3.3 70B"
PREFIX = "llama"
N_EXAMPLES = 10
res = load_ck()
res = [res[k] for k in sorted(res)]
print("claims available:", len(res))


def pos_of(nodes):
    pro = sorted(k for k in nodes if k.startswith("P"))
    con = sorted(k for k in nodes if k.startswith("N"))
    pos = {"CLAIM": (0.5, 0.9)}
    for i, k in enumerate(pro):
        pos[k] = (0.15, 0.65 - i * 0.18)
    for i, k in enumerate(con):
        pos[k] = (0.85, 0.65 - i * 0.18)
    return pos


def arrow(ax, a, b, color, rad=0.0):
    ax.add_patch(FancyArrowPatch(
        a, b, arrowstyle="-|>", mutation_scale=16, color=color,
        lw=1.8, shrinkA=22, shrinkB=22,
        connectionstyle=f"arc3,rad={rad}"))


def draw(r, fname):
    nodes = r["nodes"]
    pos = pos_of(nodes)
    sup = {tuple(e) for e in r["supports"]}
    direct = {tuple(e) for e in r["direct_attacks"]}
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(0, 1)
    ax.set_ylim(0.0, 1.0)
    ax.axis("off")
    for a, b in sup:
        arrow(ax, pos[a], pos[b], "#2e7d32")
    for a, b in direct:
        arrow(ax, pos[a], pos[b], "#c62828",
              rad=0.15 if (b, a) in direct else 0.0)
    colors = {"CLAIM": "#1565c0", "PRO": "#a5d6a7", "CON": "#ef9a9a"}
    for k, (x, y) in pos.items():
        ax.scatter(x, y, s=1400, c=colors[nodes[k]["role"]],
                   edgecolors="black", zorder=3)
        ax.text(x, y, k, ha="center", va="center", zorder=4,
                fontsize=10, fontweight="bold",
                color="white" if k == "CLAIM" else "black")
    dec = r["preferred_decisions"]
    ax.set_title(
        f"{LABEL} | {len(r['preferred_extensions'])} extension(s) | "
        f"gold={'TRUE' if r['gold'] else 'FALSE'}\n"
        f"{textwrap.fill(r['claim'], 80)}\n"
        f"credulous={dec['credulous']}  skeptical={dec['skeptical']}"
        f"  majority={dec['majority']}", fontsize=9)
    ax.text(0.0, 0.02, "green = support    red = attack "
            "(curved pair = mutual attack)", fontsize=8)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"\nClaim: {r['claim']}")
    for k in sorted(nodes):
        if k != "CLAIM":
            print(f"  {k} [{nodes[k]['role']}]: {nodes[k]['text']}")
    print("Extensions:")
    for e in r["preferred_extensions"]:
        print("  {" + ", ".join(e) + "}")
    print("saved:", fname)
    print("-" * 70)


for k in [1, 2, 4, 8]:
    found = [r for r in res if len(r["preferred_extensions"]) == k]
    if not found:
        print(f"no claim with {k} extension(s)")
        continue
    found.sort(key=lambda r: -len(r["nodes"]))
    draw(found[0], f"/kaggle/working/{PREFIX}_graph_{k}ext.png")

In [ ]:
import os, glob, shutil
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
os.makedirs("to_download", exist_ok=True)
for f in (glob.glob("*.png") + glob.glob("*.pkl")
          + glob.glob("*.csv")):
    shutil.copy(f, "to_download/" + f)
shutil.make_archive("all_results", "zip", "to_download")
print(sorted(os.listdir("to_download")))
display(FileLink("all_results.zip"))